In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united states and the capital of france"
tokens = u_tokenizer.encode(prompt)

torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_model(tokens)
sentence_meanings.shape
#sentence_meanings_with_attention_context = u_model(tokens)
#sentence_meanings_with_attention_context

torch.Size([20, 4])

In [2]:
from transformers import Gemma3ForCausalLM

gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-it")


/Users/gonul/llm-from-scratch/.llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 6879.96it/s]


In [3]:
u_model, gemma_model

(MasterModel(
   (embedding): Embedding(64, 4)
   (pos_embedding): Embedding(32, 4)
   (self_attention): SelfAttention(
     (q_weights): Linear(in_features=4, out_features=4, bias=False)
     (k_weights): Linear(in_features=4, out_features=4, bias=False)
     (v_weights): Linear(in_features=4, out_features=4, bias=False)
   )
 ),
 Gemma3ForCausalLM(
   (model): Gemma3TextModel(
     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
     (layers): ModuleList(
       (0-25): 26 x Gemma3DecoderLayer(
         (self_attn): Gemma3Attention(
           (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
           (k_proj): Linear(in_features=1152, out_features=256, bias=False)
           (v_proj): Linear(in_features=1152, out_features=256, bias=False)
           (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
           (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
           (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
         )
       

In [4]:
from plot_tokens import plot_dots
u_sentences = [
    {
        "words": sentence_meanings.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    }
]
plot_dots(u_sentences, "Models Context Space")

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

$$
Q = \text{Query (Sorgu)}
$$

$$
K = \text{Key (Anahtar)}
$$

$$
V = \text{Value (İçerik)}
$$

$$
QK^{\top} \rightarrow \text{Similarity (benzerlik)}
$$

$$
\sqrt{d_k} \rightarrow \text{Scaling (sayilar cok buyumesin)}
$$

$$
\text{softmax}(\cdot) \rightarrow \text{Attention weights}
$$

$$
\text{weights} \times V \rightarrow \text{Weighted sum}
$$

$$
d_k = \text{Key vektörünün boyutu}
$$

$$
\text{embedding\_dim} = 512
$$

$$
\text{heads} = 8
$$

$$
d_k = \frac{512}{8} = 64
$$

$$
\text{Neden } \sqrt{d_k} \text{ var?}
$$

$$
\text{Dot product büyür, softmax saturate olur (çok keskin olur).}
$$

$$
\text{Scale} = \frac{1}{\sqrt{d_k}}
$$

#q icin agirliklar(sentence i agirliklar ile carp)
#k icin agirliklar(k i agirliklar ile carp cumleden olusturcaz)
#v icin agirliklar((sentence i value agirliklar ile carp))
#q (from) -> k.v(to)
#herbir kelimenin birbirine ne kadar benzedigini anliyoruz.

In [5]:
attention_weights = sentence_meanings @ sentence_meanings.T #burda ilk sentence_meanings Q, sentence_meanings.T K oldu

In [6]:
_q_weights = torch.nn.Linear(4, 3)
_q_weights.weight, _q_weights(sentence_meanings) #bilgiyi tasiyor weightler ve bunlari egitiyoruz

(Parameter containing:
 tensor([[ 0.0104,  0.1436,  0.1804, -0.4028],
         [ 0.2416,  0.0514,  0.3449,  0.3534],
         [-0.3938,  0.4802, -0.4917,  0.2874]], requires_grad=True),
 tensor([[ 0.4420, -0.9647, -0.1709],
         [-0.1054,  0.0562, -0.4261],
         [ 0.6627, -0.4581, -0.6212],
         [ 0.0439,  0.3272, -0.4374],
         [ 0.1950,  0.0201, -0.2304],
         [ 0.1327,  0.0643,  0.2727],
         [ 0.0584,  0.5413,  0.6208],
         [-0.0341, -0.0127, -0.3652],
         [-0.2225,  0.8569, -0.9978],
         [ 0.3341,  0.7384, -1.2656],
         [ 0.1780,  0.2793, -0.1909],
         [ 0.1080,  0.2428, -0.3531],
         [ 0.1312, -0.0342,  0.1100],
         [ 0.5097,  0.0101, -1.4655],
         [ 0.0101,  0.1272, -0.7218],
         [ 0.8137,  0.5152, -1.3342],
         [ 0.1940,  0.3366, -0.4011],
         [ 0.4870,  0.2450, -0.5580],
         [ 0.1891,  0.0414,  0.0982],
         [ 0.0467,  1.0748, -1.1253]], grad_fn=<AddmmBackward0>))

In [7]:
q_weights = torch.nn.Linear(4, 3, bias=False)
k_weights = torch.nn.Linear(4, 3, bias=False)
v_weights = torch.nn.Linear(4, 3, bias=False)

q_of_sentence = q_weights(sentence_meanings)
k_of_sentence = k_weights(sentence_meanings)
v_of_sentence = v_weights(sentence_meanings)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape 


(torch.Size([20, 3]), torch.Size([20, 3]), torch.Size([20, 3]))

In [8]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores/(k_of_sentence.shape[1]**(0.5)), dim=1)

context_vector = attention_weights @ v_of_sentence
context_vector

tensor([[-0.0572,  0.0070, -0.1000],
        [-0.0945, -0.0293, -0.0778],
        [-0.0821, -0.0150, -0.0892],
        [-0.1111, -0.0266, -0.0884],
        [-0.1060, -0.0267, -0.0867],
        [-0.0942,  0.0251, -0.1402],
        [-0.1098,  0.0364, -0.1691],
        [-0.0903, -0.0171, -0.0888],
        [-0.1231, -0.0458, -0.0716],
        [-0.0753,  0.0616, -0.1717],
        [-0.1056,  0.0017, -0.1185],
        [-0.1014, -0.0050, -0.1078],
        [-0.0866,  0.0261, -0.1360],
        [-0.0541,  0.0561, -0.1484],
        [-0.0933, -0.0285, -0.0782],
        [-0.0662,  0.0800, -0.1916],
        [-0.1052, -0.0017, -0.1140],
        [-0.0894,  0.0310, -0.1446],
        [-0.0857,  0.0362, -0.1481],
        [-0.1272, -0.0215, -0.1015]], grad_fn=<MmBackward0>)

In [9]:
attention_sentences = [
    {
        "words": q_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    },
    {
        "words": k_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "blue"
    },
     {
        "words": v_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "green"
    }
]
plot_dots(attention_sentences, "Query, Key, Value Space")

In [10]:
attention_sentences = [
    {
        "words": q_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "red"
    },
    {
        "words": k_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "blue"
    },
    {
        "words": v_of_sentence.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "green"
    },
    {
        "words": context_vector.detach().numpy(),
        "labels": u_tokenizer.tokenizer(prompt),
        "color": "yellow"
    }
]
plot_dots(attention_sentences, "Query, Key, Value Space and Context Vector")

In [ ]:
# new model
# SelfAttention added to MasterModel